In [3]:
import time

# For data handling.
import jsonlines
import pandas as pd

# For visualization.
import plotly.graph_objects as go

# For fine tuning Gemini model.
import vertexai
from google import genai

# For extracting vertex experiment details.
from google.cloud import aiplatform
from google.cloud.aiplatform.metadata import context
from google.cloud.aiplatform.metadata import utils as metadata_utils
from google.genai import types
from plotly.subplots import make_subplots

# For evaluation metric computation.
from rouge_score import rouge_scorer
from tqdm import tqdm
import os

from dotenv import load_dotenv

load_dotenv()

True

In [4]:
client = genai.Client()

In [11]:
# Provide a bucket name
BUCKET_NAME = "testing-bucket-name-1234" 
BUCKET_URI = f"gs://{BUCKET_NAME}"

In [13]:
!gcloud storage buckets create gs://testing-bucket-name-1234 \
    --location=asia-southeast2

Creating gs://testing-bucket-name-1234/...


In [6]:
!gsutil ls gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_test_samples.csv
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_train_samples.jsonl
gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_val_samples.jsonl


In [7]:
!gsutil cp gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/* .

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_test_samples.csv...
Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_train_samples.jsonl...
Copying gs://github-repo/generative-ai/gemini/tuning/summarization/wikilingua/sft_val_samples.jsonl...
| [3 files][  1.6 MiB/  1.6 MiB]                                                
Operation completed over 3 objects/1.6 MiB.                                      


In [9]:
def save_jsonlines(file, instances) -> None:
    """Saves a list of json instances to a jsonlines file."""
    with jsonlines.open(file, mode="w") as writer:
        writer.write_all(instances)

def create_tuning_samples(file_path):
    """Creates tuning samples from a file."""
    with jsonlines.open(file_path) as reader:
        instances = []
        for obj in reader:
            instance = []
            for content in obj["messages"]:
                instance.append(
                    {"role": content["role"], "parts": [{"text": content["content"]}]}
                )
            instances.append({"contents": instance})
    return instances

train_file = "sft_train_samples.jsonl"
train_instances = create_tuning_samples(train_file)
len(train_instances)

# save the training instances to jsonl file
save_jsonlines(train_file, train_instances)

val_file = "sft_val_samples.jsonl"
val_instances = create_tuning_samples(val_file)
len(val_instances)

# save the validation instances to jsonl file
save_jsonlines(val_file, val_instances)

In [14]:
# Copy the tuning and evaluation data to your bucket.
!gsutil cp {train_file} {BUCKET_URI}/train/
!gsutil cp {val_file} {BUCKET_URI}/val/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://sft_train_samples.jsonl [Content-Type=application/octet-stream]...
/ [1 files][  1.2 MiB/  1.2 MiB]                                                
Operation completed over 1 objects/1.2 MiB.                                      
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying file://sft_val_samples.jsonl [Content-Type=application/octet-stream]...
/ [1 files][231.8 KiB/231.8 KiB]                                                
Operation completed over 1 objects/231.8 KiB.                               

In [15]:
# Load the test dataset using pandas as it's in the csv format.
testing_data_path = "sft_test_samples.csv"
test_data = pd.read_csv(testing_data_path)
test_data.head()

,input_text,output_text
0,Hold your arm out flat in front of you with yo...,Squeeze a line of lotion onto the tops of both...
1,"As you continue playing, surviving becomes pai...",Make a Crock Pot for better food. Create an Al...
2,Go to https://www.4kdownload.com/products/prod...,Download the 4K Video Downloader setup file. I...
3,You should know that vaginoplasty can treat a ...,Consider the health of your bladder. Find a so...
4,If you want to gather data on the frequency of...,Gather data to be graphed. Choose your range b...


In [16]:
test_data.loc[0, "input_text"]

'Hold your arm out flat in front of you with your elbow bent. The top of your forearm should form a level surface. Apply a line of lotion from the back of your hand up your arm almost to the crease of your elbow. Squeeze lotion onto both forearms.  Do not rub the lotion into your arms, rather let it sit on your arm in the line you squeezed. You can use as much or as little lotion as you feel is necessary to cover your back completely. Bend your elbows and reach both of your arms behind you, placing the lotion covered forearms against your back. Depending on how flexible you are, this may hurt a little. It might be easier to place one arm behind your back at a time. If you have shoulder pain or are not very flexible, this method may not work well for you. Rub your forearms and the backs of your hands up and down your back like windshield wipers covering as much of your back as you can. You can use your left arm first to cover your left side and then place your right arm behind and use i

In [17]:
# Article summary stats
stats = test_data["output_text"].apply(len).describe()
stats

count    100.000000
mean     186.230000
std       92.788655
min       28.000000
25%      127.250000
50%      171.000000
75%      227.000000
max      577.000000
Name: output_text, dtype: float64

In [18]:
print(f"Total `{stats['count']}` test records")
print(f"Average length is `{stats['mean']}` and max is `{stats['max']}` characters")
print("\nConsidering 1 token = 4 chars")

# Get ceil value of the tokens required.
tokens = (stats["max"] / 4).__ceil__()
print(
    f"\nSet max_token_length = stats['max']/4 = {stats['max'] / 4} ~ {tokens} characters"
)
print(f"\nLet's keep output tokens upto `{tokens}`")

Total `100.0` test records
Average length is `186.23` and max is `577.0` characters

Considering 1 token = 4 chars

Set max_token_length = stats['max']/4 = 144.25 ~ 145 characters

Let's keep output tokens upto `145`


In [20]:
# Maximum number of tokens that can be generated in the response by the LLM.
# Experiment with this number to get optimal output.
max_output_tokens = tokens

In [21]:
base_model = "gemini-2.5-flash"

In [22]:
test_doc = test_data.loc[0, "input_text"]

prompt = f"""
{test_doc}
"""

config = {
    "temperature": 0.1,
    "max_output_tokens": max_output_tokens,
}

response = client.models.generate_content(
    model=base_model,
    contents=prompt,
    config=config,
).text
print(response)

This method describes how to apply


In [23]:
# Ground truth
test_data.loc[0, "output_text"]

'Squeeze a line of lotion onto the tops of both forearms and the backs of your hands. Place your arms behind your back. Move your arms in a windshield wiper motion.'

In [24]:
# Convert the pandas dataframe to records (list of dictionaries).
corpus = test_data.to_dict(orient="records")
# Check number of records.
len(corpus)

100

In [25]:
# Create rouge_scorer object for evaluation
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

In [26]:
def run_evaluation(model, corpus: list[dict]) -> pd.DataFrame:
    """Runs evaluation for the given model and data.

    Args:
      model: The generation model.
      corpus: The test data.

    Returns:
      A pandas DataFrame containing the evaluation results.
    """
    records = []
    for item in tqdm(corpus):
        document = item.get("input_text")
        summary = item.get("output_text")

        # Catch any exception that occur during model evaluation.
        try:
            response = client.models.generate_content(
                model=model,
                contents=document,
                config=config,
            )

            # Check if response is generated by the model, if response is empty then continue to next item.
            if not (
                response
                and response.candidates
                and response.candidates[0].content.parts
            ):
                print(
                    f"\nModel has blocked the response for the document.\n Response: {response}\n Document: {document}"
                )
                continue

            # Calculates the ROUGE score for a given reference and generated summary.
            scores = scorer.score(target=summary, prediction=response.text)

            # Append the results to the records list
            records.append(
                {
                    "document": document,
                    "summary": summary,
                    "generated_summary": response.text,
                    "scores": scores,
                    "rouge1_precision": scores.get("rouge1").precision,
                    "rouge1_recall": scores.get("rouge1").recall,
                    "rouge1_fmeasure": scores.get("rouge1").fmeasure,
                    "rouge2_precision": scores.get("rouge2").precision,
                    "rouge2_recall": scores.get("rouge2").recall,
                    "rouge2_fmeasure": scores.get("rouge2").fmeasure,
                    "rougeL_precision": scores.get("rougeL").precision,
                    "rougeL_recall": scores.get("rougeL").recall,
                    "rougeL_fmeasure": scores.get("rougeL").fmeasure,
                }
            )
        except AttributeError as attr_err:
            print("Attribute Error:", attr_err)
            continue
        except Exception as err:
            print("Error:", err)
            continue
    return pd.DataFrame(records)

In [27]:
# Batch of test data.
corpus_batch = corpus[:2]

In [28]:
# Run evaluation using loaded model and test data corpus
evaluation_df = run_evaluation(base_model, corpus_batch)

100%|██████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.74s/it]


In [29]:
evaluation_df.head()

,document,summary,generated_summary,scores,rouge1_precision,rouge1_recall,rouge1_fmeasure,rouge2_precision,rouge2_recall,rouge2_fmeasure,rougeL_precision,rougeL_recall,rougeL_fmeasure
0,Hold your arm out flat in front of you with yo...,Squeeze a line of lotion onto the tops of both...,To apply lotion to your,"{'rouge1': (0.4, 0.06451612903225806, 0.111111...",0.4,0.064516,0.111111,0.0,0.0,0.0,0.4,0.064516,0.111111
1,"As you continue playing, surviving becomes pai...",Make a Crock Pot for better food. Create an Al...,To improve survival and health,"{'rouge1': (0.2, 0.05555555555555555, 0.086956...",0.2,0.055556,0.086957,0.0,0.0,0.0,0.2,0.055556,0.086957


In [30]:
evaluation_df_stats = evaluation_df.dropna().describe()

In [31]:
# Statistics of the evaluation dataframe.
evaluation_df_stats

,rouge1_precision,rouge1_recall,rouge1_fmeasure,rouge2_precision,rouge2_recall,rouge2_fmeasure,rougeL_precision,rougeL_recall,rougeL_fmeasure
count,2.000000,2.000000,2.000000,2.0,2.0,2.0,2.000000,2.000000,2.000000
mean,0.300000,0.060036,0.099034,0.0,0.0,0.0,0.300000,0.060036,0.099034
std,0.141421,0.006336,0.017080,0.0,0.0,0.0,0.141421,0.006336,0.017080
min,0.200000,0.055556,0.086957,0.0,0.0,0.0,0.200000,0.055556,0.086957
25%,0.250000,0.057796,0.092995,0.0,0.0,0.0,0.250000,0.057796,0.092995
50%,0.300000,0.060036,0.099034,0.0,0.0,0.0,0.300000,0.060036,0.099034
75%,0.350000,0.062276,0.105072,0.0,0.0,0.0,0.350000,0.062276,0.105072
max,0.400000,0.064516,0.111111,0.0,0.0,0.0,0.400000,0.064516,0.111111


In [32]:
print("Mean rougeL_precision is", evaluation_df_stats.rougeL_precision["mean"])

Mean rougeL_precision is 0.30000000000000004


In [ ]:
tuned_model_display_name = "tuning-model"
training_dataset = {
    "gcs_uri": f"{BUCKET_URI}/train/sft_train_samples.jsonl",
}

validation_dataset = types.TuningValidationDataset(
    gcs_uri=f"{BUCKET_URI}/val/sft_val_samples.jsonl"
)
# Tune a model using `tune` method.
sft_tuning_job = client.tunings.tune(
    base_model=base_model,
    training_dataset=training_dataset,
    config=types.CreateTuningJobConfig(
        tuned_model_display_name=tuned_model_display_name,
        validation_dataset=validation_dataset,
    ),
)

In [ ]:
# Get the tuning job info.
tuning_job = client.tunings.get(name=sft_tuning_job.name)
tuning_job

In [ ]:
## Cancel the tuning job
# tuning_job = client.tunings.cancel(name=sft_tuning_job.name)
# tuning_job

In [ ]:
%%time
# Wait for job completion

running_states = [
    "JOB_STATE_PENDING",
    "JOB_STATE_RUNNING",
]

while tuning_job.state.name in running_states:
    print(".", end="")
    tuning_job = client.tunings.get(name=tuning_job.name)
    time.sleep(10)
print()

In [ ]:
tuned_model = tuning_job.tuned_model.endpoint
experiment_name = tuning_job.experiment

print("Tuned model experiment", experiment_name)
print("Tuned model endpoint resource name:", tuned_model)

In [ ]:
# Locate Vertex AI Experiment and Vertex AI Experiment Run
experiment = aiplatform.Experiment(experiment_name=experiment_name)
filter_str = metadata_utils._make_filter_string(
    schema_title="system.ExperimentRun",
    parent_contexts=[experiment.resource_name],
)
experiment_run = context.Context.list(filter_str)[0]

In [ ]:
# Read data from Tensorboard
tensorboard_run_name = f"{experiment.get_backing_tensorboard_resource().resource_name}/experiments/{experiment.name}/runs/{experiment_run.name.replace(experiment.name, '')[1:]}"
tensorboard_run = aiplatform.TensorboardRun(tensorboard_run_name)
metrics = tensorboard_run.read_time_series_data()

In [ ]:
def get_metrics(metric: str = "/train_total_loss"):
    """Get metrics from Tensorboard.

    Args:
      metric: metric name, eg. /train_total_loss or /eval_total_loss.

    Returns:
      steps: list of steps.
      steps_loss: list of loss values.
    """
    loss_values = metrics[metric].values
    steps_loss = []
    steps = []
    for loss in loss_values:
        steps_loss.append(loss.scalar.value)
        steps.append(loss.step)
    return steps, steps_loss

In [ ]:
# Get Train and Eval Loss
train_loss = get_metrics(metric="/train_total_loss")
eval_loss = get_metrics(metric="/eval_total_loss")

In [ ]:
# Plot the train and eval loss metrics using Plotly python library

fig = make_subplots(
    rows=1, cols=2, shared_xaxes=True, subplot_titles=("Train Loss", "Eval Loss")
)

# Add traces
fig.add_trace(
    go.Scatter(x=train_loss[0], y=train_loss[1], name="Train Loss", mode="lines"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(x=eval_loss[0], y=eval_loss[1], name="Eval Loss", mode="lines"),
    row=1,
    col=2,
)

# Add figure title
fig.update_layout(title="Train and Eval Loss", xaxis_title="Steps", yaxis_title="Loss")

# Set x-axis title
fig.update_xaxes(title_text="Steps")

# Set y-axes titles
fig.update_yaxes(title_text="Loss")

# Show plot
fig.show()

In [ ]:
if True:
    # Test with the loaded model.
    print("***Testing***")
    print(
        client.models.generate_content(
            model=tuned_model, contents=prompt, config=config
        ).text
    )
else:
    print("State:", tuning_job.state.name.state)
    print("Error:", tuning_job.state.name.error)

In [ ]:
# run evaluation
evaluation_df_post_tuning = run_evaluation(tuned_model, corpus_batch)

In [ ]:
evaluation_df_post_tuning.head()

In [ ]:
evaluation_df_post_tuning_stats = evaluation_df_post_tuning.dropna().describe()

In [ ]:
# Statistics of the evaluation dataframe post model tuning.
evaluation_df_post_tuning_stats

In [ ]:
print(
    "Mean rougeL_precision is", evaluation_df_post_tuning_stats.rougeL_precision["mean"]
)

In [ ]:
improvement = round(
    (
        (
            evaluation_df_post_tuning_stats.rougeL_precision["mean"]
            - evaluation_df_stats.rougeL_precision["mean"]
        )
        / evaluation_df_stats.rougeL_precision["mean"]
    )
    * 100,
    2,
)
print(
    f"Model tuning has improved the rougeL_precision by {improvement}% (result might differ based on each tuning iteration)"
)

In [ ]:
# Delete Experiment.
delete_experiments = True
if delete_experiments:
    experiments_list = aiplatform.Experiment.list()
    for experiment in experiments_list:
        if experiment.resource_name == experiment_name:
            print(experiment.resource_name)
            experiment.delete()
            break

print("***" * 10)

# Delete Endpoint.
delete_endpoint = True
# If force is set to True, all deployed models on this
# Endpoint will be first undeployed.
if delete_endpoint:
    for endpoint in aiplatform.Endpoint.list():
        if endpoint.resource_name == tuned_model:
            print(endpoint.resource_name)
            endpoint.delete(force=True)
            break

print("***" * 10)

# Delete Cloud Storage Bucket.
delete_bucket = True
if delete_bucket:
    ! gsutil -m rm -r $BUCKET_URI